In [1]:
from dotenv import load_dotenv
import os
from pathlib import Path
from openai import AzureOpenAI
from embeddings import UserDocumentCollection
import os, json, re
import pandas as pd

import numpy as np
from pypdf import PdfReader
import win32com.client as win32
from time import sleep

In [2]:
load_dotenv()

True

In [4]:
data_folder = os.environ.get("FOLDER_PATH")
output_folder = Path(data_folder).parent / "results"

In [6]:
data = pd.read_excel(Path(data_folder)/ "PC_Biotech Act 12112025.xlsx",
                     sheet_name = "PC_all", skiprows=1)

In [11]:
pd.set_option("display.max_columns", None)
for i, col in enumerate(data.columns):
    print(i, col)


0   
1 Feedbackreference
2 Datefeedback
3 Usertype
4 Tr_number
5 Organization
6 Scope
7 Governancelevel
8 Companysize
9 Country
10 Firstname
11 Surname
12 Email
13 Publicationstatus
14 Feedbacklanguage
15 Attachment
16 Status
17 You have identified yourself as a business association or a company/business. Please indicate whether you belong to one of the following areas:
18 Do you identify yourself as a private investor (e.g. venture capitalist, business angel)?
19 You have identified yourself as a private investor in the biotechnology sector. Please indicate the type of investment you provide:
20 Are you or the organisation you represent part of a cluster or of a cluster organisation? 


 'Clusters are groups of firms, related economic actors, and institutions located near each other and with sufficient scale to develop specialised expertise, services, resources, suppliers and skills.' [link to definition of clusters]

 
 'Cluster organisations are the legal entities that support the s

In [13]:
closed_vals = {
    "strongly agree",
    "agree",
    "disagree",
    "strongly disagree",
    "neutral",
    "yes",
    "no",
}

open_text_cols = []

for col in data.columns[23:]:
    s = data[col].dropna().astype(str).str.lower().str.strip()
    # remove simple trailing punctuation like "Yes." or "No!"
    s_norm = s.str.replace(r"[.!?]+$", "", regex=True).str.strip()
    
    # if NO cell is exactly one of the closed_vals -> keep as open text column
    if not s_norm.isin(closed_vals).any():
        open_text_cols.append(col)

open_text_df = data[open_text_cols]

for i, col in enumerate(open_text_df.columns):
    print(i, col)

0 1.Q1. You are asked the following question as you are replying on behalf of a company/business.

What has been the approximate share of your global revenues generated in the EU in the last two years (2023-2024)?
1 1.Q1a. If you would like to explain why you are not present in the EU market, you can do so here:
2 2.Q2. Please indicate other phases of the innovation and manufacturing cycle where there are regulatory barriers caused by EU rules.
3 2.Q3. Please substantiate your statements with additional evidence on the challenges resulting from the EU regulatory environment.
4 2.Q4. In your view, what actions at EU level are necessary to improve the regulatory environment for biotechnology and biomanufacturing in the EU? Please substantiate your statements with views and evidence on the ways forward.
5 2.Q5a. Regarding predictability: Please indicate the reasons why, and in which third-country(ies) this applies.
6 2.Q5b. Regarding complexity and clarity: Please indicate the reasons why

In [21]:
len(data["Feedbackreference"][1])

9

In [17]:
for row in data["Feedbackreference"]:
    print(row)

F33080045
F33102384
F33114470
F33114515
F3615331
F33093302
F33114872
F33114464
F33114459
F33114469
F33091444
F33113313
F33114712
F33109043
F33116816
F33115063
F33114514
F33114579
F33090072
F33114552
F33114948
F3590009
F3593278
F33094210
F33091336
F33114287
F33079531
F33115326
F33115935
F33118010
F33114478
F33114701
F33114475
F33114961
F33114484
F33114933
F33113073
F3593805
F33113178
F33114528
F33114703
F33113159
F33113271
F33114265
F33114320
F33114529
F33114966
F33114580
F33113897
F33114518
F33114543
F33114380
F33114947
F33114601
F33114504
F33068713
F33114346
F33114350
F33114480
F33114525
F33114705
F33114719
F33114925
F33118015
F3592557
F33114463
F33068708
F33114949
F3590576
F33101427
F33114556
F33118013
F33114493
F33074482
F3589649
F3590596
F3590655
F3653902
F3594915
F33070967
F3592098
F33114498
F33114591
F3643103
F33114282
F33114603
F33114501
F33079709
F33114476
F33114519
F33116244
F33114508
F33114522
F3590175
F33113287
F33114358
F33114491
F33114510
F33114523
F33117135
F3592348
F3655

In [26]:
for col in data.columns:
    print(data[col][1])

Biotechact
F33102384
2025/10/28 17:15:41
Company/business
nan
bYoRNA
nan
nan
Small (< 50 employees)
France
nan
nan
nan
The feedback can be published in an anonymous way
French
nan
Published
Company conducting research and/or development in biotechnology and/or biomanufacturing | Biotechnology manufacturer
No
nan
Yes
Medical/pharmaceutical | Agricultural | Industrial | Bioinformatics | Biotechnology for defence and security
nan
Strongly agree
Strongly agree
Strongly agree
Strongly agree
Disagree
Neutral
No revenues generated in the EU
Enterprise at the R & D stage not yet generating income
Disagree
Agree
Strongly agree
Strongly agree
Agree
Agree
Neutral
Neutral
nan
The introduction of biomedicaments (from biotechnology) comes within a rather strict regulatory framework, which greatly increases the development costs and reduces the placing on the market. In the context of innovative cell & gene therapy products for niche markets (low number of patients), this makes the business model com

In [33]:
open_text_cols2=[]

for col in data.columns[23:]:

    count_long_row = 0
    i = 0
    column = data[col].dropna().astype(str).str.lower().str.strip()
    lengths = column.apply(lambda x: len(x)).to_list()

    while i < len(lengths):
        if lengths[i] > 50:
            count_long_row += 1
        
        if count_long_row == 1:
            open_text_cols2.append(col)
            break
        else:
            i += 1


columns_of_interest = ["Feedbackreference","Usertype"] + open_text_cols2 
open_text_df = data[columns_of_interest]

for i, col in enumerate(open_text_df.columns):
    print(i, col)

0 Feedbackreference
1 Usertype
2 1.Q1a. If you would like to explain why you are not present in the EU market, you can do so here:
3 2.Q2. Please indicate other phases of the innovation and manufacturing cycle where there are regulatory barriers caused by EU rules.
4 2.Q3. Please substantiate your statements with additional evidence on the challenges resulting from the EU regulatory environment.
5 2.Q4. In your view, what actions at EU level are necessary to improve the regulatory environment for biotechnology and biomanufacturing in the EU? Please substantiate your statements with views and evidence on the ways forward.
6 2.Q5a. Regarding predictability: Please indicate the reasons why, and in which third-country(ies) this applies.
7 2.Q5b. Regarding complexity and clarity: Please indicate the reasons why, and in which third-country(ies) this applies.
8 2.Q5c. Regarding compliance costs: Please indicate the reasons why, and in which third-country(ies) this applies.
9 2.Q5d. Regarding 

In [34]:
len(open_text_df.columns)

50

## remove columns 7.Q2.a , 7.Q4.a

In [37]:
drop_list = []
for column in open_text_df.columns:
    if column.startswith("7.Q2a") or column.startswith("7.Q4a"):
        drop_list.append(column)

        

In [39]:
drop_list

['7.Q2a. What are the main reasons for relying on data sourced from outside of the EU/EEA?',
 '7.Q4a. In what capacity does your organisation expect to be involved in the European Health Data Space? Please select the capacity(ies) that is/are most relevant for you.']

In [40]:
open_text_df = open_text_df.drop(columns=drop_list)
len(open_text_df.columns)

48

In [42]:
df = open_text_df.copy()

In [47]:
df.groupby("Usertype").size()


Usertype
Academic/Research Institution          44
Business Association                   61
Company/business                       91
EU Citizen                             54
NGO (Non-governmental organisation)    47
Non-EU Citizen                          9
Other                                  30
Public authority                       21
Trade Union                             2
dtype: int64

In [52]:
prompt = """
You are an AI assistant helping to summarise open-text responses from a public consultation 
conducted for a European Commission policy initiative on biotechnology.

Your task is to summarise the meaningful open-text responses provided by the user type indicated below.
The summary must reflect what respondents actually said, in relation to the question asked.

Rules for handling responses:
- Ignore meaningless responses (e.g., RAS, NA, N/A, none, empty strings, or entries made only of 
  punctuation, digits, or random characters).
- If **no meaningful responses** remain, answer:
  "No respondents from this user group provided a substantive answer to this question."
- If **one meaningful response** remains, summarise it faithfully in 1–2 sentences.
- If **two or three meaningful responses** remain, summarise them in 2–3 sentences.
- If **four or more meaningful responses** exist, summarise them concisely in 3–5 sentences.
- The summary must be written in clear, neutral British English.
- Do not invent or infer content that is not present in the responses.
- Do not quote respondents; synthesise the ideas smoothly.
- Focus on summarising the answers **in relation to the question**, not on identifying themes.

User type: {Usertype}
Question: "{question}"

Open-text responses:
{responses}

Please provide only the summary, with no introduction or meta-commentary.
"""


In [53]:
client = AzureOpenAI(
    azure_endpoint=os.environ["AZURE_COGNITIVE_SERVICES_ENDPOINT"],
    api_key=os.environ["AZURE_COGNITIVE_API"],
    api_version="2024-10-21",  # 2024-02-01+ supports data_sources
)

In [74]:
def get_summary(user_type, question, responses):
    """
    user_type: str
    question: str
    responses: str OR list of strings (see note below)
    """
    
    final_prompt = prompt.format(
        Usertype=user_type,
        question=question,
        responses=responses,
    )

    # 2. DEBUG: print exactly what is sent to the model
    print("======== PROMPT SENT TO MODEL ========")
    print(final_prompt)
    print("======================================")

    # 3. Call the API with that prompt
    completion = client.chat.completions.create(
        model="gpt-4.1",  # your real model here
        messages=[
            {
                "role": "user",
                "content": final_prompt,
            }
        ],
    )
         

    summary = completion.choices[0].message.content or ""
    return summary


In [75]:
import pandas as pd
import json
from tqdm import tqdm

summaries = {}   

In [76]:


# all question columns: SAME FOR EVERY USERTYPE
question_cols = [c for c in df.columns if c not in ("Usertype", "Feedbackreference")]

summaries = {}

for user_type in df["Usertype"].dropna().unique():
    summaries[user_type] = {}

    df_usertype = df[df["Usertype"] == user_type]

    # ONE BAR PER USERTYPE, TOTAL = NUMBER OF QUESTIONS
    for col in tqdm(question_cols,
                    desc=f"{user_type} questions",
                    total=len(question_cols)):

        responses = df_usertype[col].dropna().astype(str).str.strip()

        # your filter, unchanged
        meaningful_responses = responses[
            ~responses.str.lower().isin({"ras", "na", "n/a", "", ".", ",", "-", "_"})
        ]

        if len(meaningful_responses) == 0:
            summary = (
                "No respondents from this user group provided a substantive answer to this question."
            )
        else:
            text = "\n - ".join(meaningful_responses.tolist())
            summary = get_summary(user_type, col, text)

        summaries[user_type][col] = summary

    # checkpoint per user type, in output_folder
    safe_user_type = user_type.replace("/", "_").replace("\\", "_")
    filename = f"checkpoint_{safe_user_type}.json"
    with open(Path(output_folder)/filename, "w", encoding="utf-8") as f:
        json.dump(summaries[user_type], f, ensure_ascii=False, indent=2)


Company/business questions:   0%|          | 0/46 [00:00<?, ?it/s]

======== PROMPT SENT TO MODEL ========

You are an AI assistant helping to summarise open-text responses from a public consultation 
conducted for a European Commission policy initiative on biotechnology.

Your task is to summarise the meaningful open-text responses provided by the user type indicated below.
The summary must reflect what respondents actually said, in relation to the question asked.

Rules for handling responses:
- Ignore meaningless responses (e.g., RAS, NA, N/A, none, empty strings, or entries made only of 
  punctuation, digits, or random characters).
- If **no meaningful responses** remain, answer:
  "No respondents from this user group provided a substantive answer to this question."
- If **one meaningful response** remains, summarise it faithfully in 1–2 sentences.
- If **two or three meaningful responses** remain, summarise them in 2–3 sentences.
- If **four or more meaningful responses** exist, summarise them concisely in 3–5 sentences.
- The summary must be wri

Company/business questions:   2%|▏         | 1/46 [00:01<01:13,  1.64s/it]

======== PROMPT SENT TO MODEL ========

You are an AI assistant helping to summarise open-text responses from a public consultation 
conducted for a European Commission policy initiative on biotechnology.

Your task is to summarise the meaningful open-text responses provided by the user type indicated below.
The summary must reflect what respondents actually said, in relation to the question asked.

Rules for handling responses:
- Ignore meaningless responses (e.g., RAS, NA, N/A, none, empty strings, or entries made only of 
  punctuation, digits, or random characters).
- If **no meaningful responses** remain, answer:
  "No respondents from this user group provided a substantive answer to this question."
- If **one meaningful response** remains, summarise it faithfully in 1–2 sentences.
- If **two or three meaningful responses** remain, summarise them in 2–3 sentences.
- If **four or more meaningful responses** exist, summarise them concisely in 3–5 sentences.
- The summary must be wri

Company/business questions:   4%|▍         | 2/46 [00:04<01:47,  2.44s/it]

...

======== PROMPT SENT TO MODEL ========

You are an AI assistant helping to summarise open-text responses from a public consultation 
conducted for a European Commission policy initiative on biotechnology.

Your task is to summarise the meaningful open-text responses provided by the user type indicated below.
The summary must reflect what respondents actually said, in relation to the question asked.

Rules for handling responses:
- Ignore meaningless responses (e.g., RAS, NA, N/A, none, empty strings, or entries made only of 
  punctuation, digits, or random characters).
- If **no meaningful responses** remain, answer:
  "No respondents from this user group provided a substantive answer to this question."
- If **one meaningful response** remains, summarise it faithfully in 1–2 sentences.
- If **two or three meaningful responses** remain, summarise them in 2–3 sentences.
- If **four or more meaningful responses** exist, summarise them concisely in 3–5 sentences.
- The summary must be wri

Company/business questions:  11%|█         | 5/46 [00:15<02:19,  3.40s/it]

======== PROMPT SENT TO MODEL ========

You are an AI assistant helping to summarise open-text responses from a public consultation 
conducted for a European Commission policy initiative on biotechnology.

Your task is to summarise the meaningful open-text responses provided by the user type indicated below.
The summary must reflect what respondents actually said, in relation to the question asked.

Rules for handling responses:
- Ignore meaningless responses (e.g., RAS, NA, N/A, none, empty strings, or entries made only of 
  punctuation, digits, or random characters).
- If **no meaningful responses** remain, answer:
  "No respondents from this user group provided a substantive answer to this question."
- If **one meaningful response** remains, summarise it faithfully in 1–2 sentences.
- If **two or three meaningful responses** remain, summarise them in 2–3 sentences.
- If **four or more meaningful responses** exist, summarise them concisely in 3–5 sentences.
- The summary must be wri

Company/business questions:  11%|█         | 5/46 [00:19<02:42,  3.96s/it]


KeyboardInterrupt: 

In [67]:
summary_df = pd.DataFrame.from_dict(summaries, orient="index")

summary_df = summary_df.reset_index().rename(columns={"index": "Usertype"})
user_counts = df.groupby("Usertype").size()   # Series: index = Usertype, value = count
summary_df["number_of_users"] = summary_df["Usertype"].map(user_counts)
cols = summary_df.columns.tolist()
new_order = ["Usertype", "number_of_users"] + [c for c in cols if c not in ("Usertype", "number_of_users")]

summary_df = summary_df[new_order]


In [68]:
summary_df.head()

,Usertype,number_of_users,"1.Q1a. If you would like to explain why you are not present in the EU market, you can do so here:",2.Q2. Please indicate other phases of the innovation and manufacturing cycle where there are regulatory barriers caused by EU rules.,2.Q3. Please substantiate your statements with additional evidence on the challenges resulting from the EU regulatory environment.,"2.Q4. In your view, what actions at EU level are necessary to improve the regulatory environment for biotechnology and biomanufacturing in the EU? Please substantiate your statements with views and evidence on the ways forward.","2.Q5a. Regarding predictability: Please indicate the reasons why, and in which third-country(ies) this applies.","2.Q5b. Regarding complexity and clarity: Please indicate the reasons why, and in which third-country(ies) this applies.","2.Q5c. Regarding compliance costs: Please indicate the reasons why, and in which third-country(ies) this applies.","2.Q5d. Regarding speed of reaching the market: Please indicate the reasons why, and in which third-country(ies) this applies.","2.Q5e. Regarding the level of safety and security: Please indicate the reasons why, and in which third-country(ies) this applies.",2.Q6. Please indicate any other relevant factors that characterise the regulations in non-EU countries and that are applicable to biotechnology and biomanufacturing products.,3.Q3a. Please indicate other relevant private and public financial instruments.,3.Q5. Please indicate other factors that drive investment in a biotechnology and/or biomanufacturing company here.,"3.Q6a. If you would like to indicate why, you can do so here.","3.Q7a. If you would like to indicate why, you can do so here.",3.Q8. Please substantiate your statements with additional evidence on the challenges related to access to finance in the EU.,"3.Q9. In your view, what actions at EU level are necessary for the public sector to attract/derisk private investments in biotechnology and/or biomanufacturing? Please substantiate your statements with views and evidence on the ways forward. \n\nYou can provide references of successful schemes existing at EU level, national level or in other jurisdictions to attract private capital in biotechnology.","3.Q10. In your view, what actions at EU level are necessary to prioritise funding for high-risk and high-reward biotechnology research and innovation? Please substantiate your statements with views and evidence on the ways forward.","3.Q11. In your view, what other actions are necessary at EU level? Please substantiate your statements with views and evidence on the ways forward.",4.Q2. Please indicate other factors impacting biotechnology clusters and/or cluster organisations in the EU.,4.Q3. Please substantiate your statements with additional evidence on the challenges faced by biotechnology clusters and/or cluster organisations in the EU.,"4.Q4. In your view, what actions at EU level are necessary to enhance the impact of biotechnology clusters and/or cluster organisations in the EU? Please substantiate your statements with views and evidence on the ways forward.","4.Q5. In your view, what actions at EU level are necessary to create more synergies between existing clusters and/or cluster organisations and facilitate pooling of expertise and resources in the EU? Please substantiate your statements with views and evidence on the ways forward here.",5.Q2. Please indicate other challenges impacting biotechnology manufacturing in the EU.,5.Q3. Please substantiate your statements with additional evidence on the challenges impacting biotechnology manufacturing in the EU.,"5.Q4. In your view, what actions at EU level are necessary to enhance the impact of biotechnology manufacturing in the EU? Please substantiate your statements with views and evidence on the ways forward.",6.Q2. Please indicate other challenges faced by the workforce for biotechnology in the EU.,6.Q4. Please indicate other factors leading to the EU

In [69]:
summary_df.to_excel(output_folder / "PC_opentext_questions_usertype_summaries.xlsx", index=False)